### Session Creation

In [1]:
import os
import sys

python_path = sys.executable

os.environ["PYSPARK_PYTHON"] = python_path
os.environ["PYSPARK_DRIVER_PYTHON"] = python_path

os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

print("Python used by PySpark:")
print(python_path)

Python used by PySpark:
d:\Projects\Log-Analysis\.venv\Scripts\python.exe


In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("NASA Log Analysis")
    .master("local[*]")
    .config("spark.pyspark.python", python_path)
    .config("spark.pyspark.driver.python", python_path)
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.1.2


In [3]:
log_df = spark.read.text("data/access.log")

log_df.printSchema()

root
 |-- value: string (nullable = true)



In [7]:
first = log_df.first()
print(first)
print(first['value'])


Row(value='in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0400] "GET /shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0" 200 1839')
in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0400] "GET /shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0" 200 1839


In [8]:
rows = log_df.take(5)

for i in rows:
    print(i['value'])

in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0400] "GET /shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0" 200 1839
uplherc.upl.com - - [01/Aug/1995:00:00:07 -0400] "GET / HTTP/1.0" 304 0
uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/ksclogo-medium.gif HTTP/1.0" 304 0
uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/MOSAIC-logosmall.gif HTTP/1.0" 304 0
uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/USA-logosmall.gif HTTP/1.0" 304 0


In [9]:
rows = log_df.limit(5).collect()

for row in rows:
    print(row['value'])

in24.inetnebr.com - - [01/Aug/1995:00:00:01 -0400] "GET /shuttle/missions/sts-68/news/sts-68-mcc-05.txt HTTP/1.0" 200 1839
uplherc.upl.com - - [01/Aug/1995:00:00:07 -0400] "GET / HTTP/1.0" 304 0
uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/ksclogo-medium.gif HTTP/1.0" 304 0
uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/MOSAIC-logosmall.gif HTTP/1.0" 304 0
uplherc.upl.com - - [01/Aug/1995:00:00:08 -0400] "GET /images/USA-logosmall.gif HTTP/1.0" 304 0


In [10]:
print("Original Data : ")
log_df.show(5, truncate=True)

Original Data : 
+--------------------+
|               value|
+--------------------+
|in24.inetnebr.com...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
+--------------------+
only showing top 5 rows


In [11]:
print("\n Using select : ")
log_df.select('value').show(5)

print(rows[1]['value'])


 Using select : 
+--------------------+
|               value|
+--------------------+
|in24.inetnebr.com...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
+--------------------+
only showing top 5 rows
uplherc.upl.com - - [01/Aug/1995:00:00:07 -0400] "GET / HTTP/1.0" 304 0


In [12]:
print("Print new columns : ")
new_df = log_df.withColumnRenamed('value','log')
new_df.show(5)

print("Column Names : ")
print(new_df.columns)

print("\nOld Name : ")
print(log_df.columns)

Print new columns : 
+--------------------+
|                 log|
+--------------------+
|in24.inetnebr.com...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
|uplherc.upl.com -...|
+--------------------+
only showing top 5 rows
Column Names : 
['log']

Old Name : 
['value']


In [13]:
log_df.columns
log_df.rollup

<bound method DataFrame.rollup of DataFrame[value: string]>

In [17]:
from pyspark.sql.functions import regexp_extract, col
pattern = r'(\S+) (\S+) (\S+) \[([^\]]+)\] "(\S+) (\S+) (\S+)" (\d+) (\S+)'

# Extract individual fields
parsed_df = log_df.select(
    regexp_extract("value", pattern, 1).alias("host"),
    regexp_extract("value", pattern, 2).alias("ident"),
    regexp_extract("value", pattern, 3).alias("authuser"),
    regexp_extract("value", pattern, 4).alias("timestamp"),
    regexp_extract("value", pattern, 5).alias("method"),
    regexp_extract("value", pattern, 6).alias("url"),
    regexp_extract("value", pattern, 7).alias("protocol"),
    regexp_extract("value", pattern, 8).cast("int").alias("status"),
    regexp_extract("value", pattern, 9).cast("long").alias("bytes")
)

# Display the result
parsed_df.show(10, truncate=False)

+---------------------------+-----+--------+--------------------------+------+-----------------------------------------------+--------+------+-----+
|host                       |ident|authuser|timestamp                 |method|url                                            |protocol|status|bytes|
+---------------------------+-----+--------+--------------------------+------+-----------------------------------------------+--------+------+-----+
|in24.inetnebr.com          |-    |-       |01/Aug/1995:00:00:01 -0400|GET   |/shuttle/missions/sts-68/news/sts-68-mcc-05.txt|HTTP/1.0|200   |1839 |
|uplherc.upl.com            |-    |-       |01/Aug/1995:00:00:07 -0400|GET   |/                                              |HTTP/1.0|304   |0    |
|uplherc.upl.com            |-    |-       |01/Aug/1995:00:00:08 -0400|GET   |/images/ksclogo-medium.gif                     |HTTP/1.0|304   |0    |
|uplherc.upl.com            |-    |-       |01/Aug/1995:00:00:08 -0400|GET   |/images/MOSAIC-logosmall.gif

In [4]:
parsed_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("data/nasa_logs_csv")

NameError: name 'parsed_df' is not defined

In [20]:
import os

print("Current directory:", os.getcwd())
print("Output path:", os.path.abspath("data/nasa_logs_csv"))

Current directory: d:\Projects\Log-Analysis
Output path: d:\Projects\Log-Analysis\data\nasa_logs_csv


In [21]:
os.makedirs("output", exist_ok=True)

parsed_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/nasa_logs_csv")

Py4JJavaError: An error occurred while calling o161.csv.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: D:\Software\hadoop-3.5.0\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdir

In [5]:
test_df = spark.createDataFrame(
    [(1, "hello"), (2, "NASA")],
    ["id", "name"]
)

test_df.show()

test_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output/test_csv")

+---+-----+
| id| name|
+---+-----+
|  1|hello|
|  2| NASA|
+---+-----+



Py4JJavaError: An error occurred while calling o65.csv.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: D:\Software\hadoop-3.5.0\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs

In [7]:
try:
    test_df.write \
        .mode("overwrite") \
        .option("header", True) \
        .csv("output/test_csv")
except Exception as e:
    print(e)

An error occurred while calling o78.csv.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: D:\Software\hadoop-3.5.0\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1179)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:861)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:901)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:873)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:900)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java

In [8]:
print(spark.version)

4.1.2


In [9]:
spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
spark.version

'4.1.2'

In [10]:
print("Spark:", spark.version)

print(
    "Hadoop:",
    spark.sparkContext._jvm.org.apache.hadoop.util.VersionInfo.getVersion()
)

Spark: 4.1.2
Hadoop: 3.4.2
